In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from PIL import Image, ImageChops, ImageEnhance
from sklearn.model_selection import train_test_split
from io import BytesIO

# --- 0. GPU KONTROLÜ ---
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU AKTİF! Kaggle sunucularında eğitim başlıyor.")
else:
    print("⚠️ GPU BULUNAMADI! Sağ panelden Accelerator olarak GPU (T4x2) seçtiğinden emin ol.")

# --- 1. RAM DOSTU ELA İŞLEYİCİ ---
class CasiaELAGenerator(Sequence):
    def __init__(self, file_paths, labels, batch_size=32, target_size=(299, 299), ela_quality=90):
        self.file_paths = file_paths
        self.labels = labels
        self.batch_size = batch_size
        self.target_size = target_size
        self.ela_quality = ela_quality

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / self.batch_size))

    def _generate_ela_array(self, image_path):
        try:
            original = Image.open(image_path).convert('RGB')
            buffer = BytesIO()
            original.save(buffer, format='JPEG', quality=self.ela_quality)
            buffer.seek(0)
            compressed = Image.open(buffer)
            
            ela_image = ImageChops.difference(original, compressed)
            extrema = ela_image.getextrema()
            max_diff = max([ex[1] for ex in extrema]) if extrema else 1
            if max_diff == 0: max_diff = 1
            scale = 255.0 / max_diff
            ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
            
            ela_image = ela_image.resize(self.target_size)
            img_array = np.array(ela_image) / 255.0
            
            return img_array
        except Exception as e:
            # Bozuk/okunamayan resimler için boş array dön (eğitimin çökmesini engeller)
            return np.zeros((self.target_size[0], self.target_size[1], 3))

    def __getitem__(self, idx):
        batch_x = self.file_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_y = self.labels[idx * self.batch_size : (idx + 1) * self.batch_size]
        x = [self._generate_ela_array(path) for path in batch_x]
        return np.array(x), np.array(batch_y)

# --- 2. VERİ YOLLARININ HAZIRLANMASI (CASIA'YA ÖZEL) ---
def prepare_casia_paths(base_path):
    file_paths, labels = [], []
    
    # Kaggle input dizinini tarayalım
    for root, dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root).lower()
        
        # Sadece resim dosyalarını alalım
        for filename in files:
            lower_name = filename.lower()
            if lower_name.endswith(('.jpg', '.jpeg', '.png', '.tif', '.bmp')):
                full_path = os.path.join(root, filename)
                
                # CASIA'da Authentic (Orijinal) klasörleri genellikle 'au' içerir
                if folder_name == 'au' or 'authentic' in folder_name:
                    file_paths.append(full_path)
                    labels.append(0)
                # CASIA'da Tampered (Sahte) klasörleri genellikle 'tp' içerir
                elif folder_name == 'tp' or 'tampered' in folder_name:
                    file_paths.append(full_path)
                    labels.append(1)
                    
    return file_paths, labels

# --- MAIN EĞİTİM AKIŞI ---
# Kaggle'a eklenen tüm veri setleri /kaggle/input/ altında toplanır
KAGGLE_INPUT_DIR = "/kaggle/input" 
BATCH_SIZE = 32  # GPU T4x2 için ideal
EPOCHS_PHASE_1 = 5  # Sadece dış katman (Isınma)
EPOCHS_PHASE_2 = 10 # İnce Ayar (Fine-Tuning)

print("\nVeri yolları taranıyor...")
X_paths, y_labels = prepare_casia_paths(KAGGLE_INPUT_DIR)

print(f"Bulunan Orijinal (Au) Görsel: {y_labels.count(0)}")
print(f"Bulunan Sahte (Tp) Görsel: {y_labels.count(1)}")
print(f"Toplam görsel sayısı: {len(X_paths)}")

if len(X_paths) == 0:
    raise ValueError("HATA: Görseller bulunamadı! Sağ panelden 'Add Input' diyerek CASIA veri setini eklediğinden emin ol.")

# Veriyi bölme (Eğitim %80, Validasyon %20)
X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    X_paths, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# Generator'ları oluştur (Veri çoğaltma kaldırıldı)
train_generator = CasiaELAGenerator(X_train_paths, y_train, batch_size=BATCH_SIZE)
val_generator = CasiaELAGenerator(X_val_paths, y_val, batch_size=BATCH_SIZE)

# --- FAZ 1: TRANSFER LEARNING (SADECE ÜST KATMANLAR) ---
print("\n--- FAZ 1: Üst Katmanların Eğitimi Başlıyor ---")
base_model = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS_PHASE_1)

# --- FAZ 2: İNCE AYAR (FINE-TUNING) ---
print("\n--- FAZ 2: İnce Ayar (Fine-Tuning) Başlıyor ---")
base_model.trainable = True

# Son 30 katmanı eğitime aç, gerisini dondur ki ImageNet ezberini bozmayalım
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Çok daha küçük bir öğrenme oranı (1e-5) kullanıyoruz
model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

save_path = '/kaggle/working/best_casia_ela_model.h5'
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    save_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_PHASE_2,
    callbacks=[checkpoint]
)

print(f"\n✅ Eğitim tamamlandı! En iyi model '{save_path}' adresine kaydedildi.")
print("Kaggle sağ panelinden 'Output' sekmesini açarak modeli indirebilirsin.")

✅ GPU AKTİF! Kaggle sunucularında eğitim başlıyor.

Veri yolları taranıyor...
Bulunan Orijinal (Au) Görsel: 7491
Bulunan Sahte (Tp) Görsel: 5123
Toplam görsel sayısı: 12614

--- FAZ 1: Üst Katmanların Eğitimi Başlıyor ---
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1779576134.964982     140 service.cc:152] XLA service 0x7c7178811690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779576134.965020     140 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779576134.965024     140 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779576136.312284     140 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-23 22:42:23.200546: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.78 = (f32[32,128,147,147]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,147,147]{3,2,1,0} %bitcast.7564, f32[128,1,3,3]{3,2,1,0} %bitcast.7568), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=128, custom_call_target="__cudnn$convForward", metadata={op_type="DepthwiseConv2dNative" op_name="functional_1/block2_sepconv2_1/separable_conv2d/depthwise" 

184/316 ━━━━━━━━━━━━━━━━━━━━ 1:38 743ms/step - accuracy: 0.7898 - loss: 0.4456

...

316/316 ━━━━━━━━━━━━━━━━━━━━ 251s 698ms/step - accuracy: 0.8183 - loss: 0.3803 - val_accuracy: 0.8692 - val_loss: 0.3024
Epoch 2/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.8913 - loss: 0.2514
Epoch 2: val_accuracy improved from 0.86920 to 0.87515, saving model to /kaggle/working/best_casia_ela_model.h5


316/316 ━━━━━━━━━━━━━━━━━━━━ 184s 581ms/step - accuracy: 0.8913 - loss: 0.2513 - val_accuracy: 0.8751 - val_loss: 0.2887
Epoch 3/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 461ms/step - accuracy: 0.9153 - loss: 0.1998
Epoch 3: val_accuracy improved from 0.87515 to 0.88149, saving model to /kaggle/working/best_casia_ela_model.h5


316/316 ━━━━━━━━━━━━━━━━━━━━ 182s 576ms/step - accuracy: 0.9153 - loss: 0.1998 - val_accuracy: 0.8815 - val_loss: 0.2770
Epoch 4/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 461ms/step - accuracy: 0.9363 - loss: 0.1580
Epoch 4: val_accuracy improved from 0.88149 to 0.88426, saving model to /kaggle/working/best_casia_ela_model.h5


316/316 ━━━━━━━━━━━━━━━━━━━━ 182s 575ms/step - accuracy: 0.9363 - loss: 0.1580 - val_accuracy: 0.8843 - val_loss: 0.2800
Epoch 5/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step - accuracy: 0.9489 - loss: 0.1324
Epoch 5: val_accuracy improved from 0.88426 to 0.88823, saving model to /kaggle/working/best_casia_ela_model.h5


316/316 ━━━━━━━━━━━━━━━━━━━━ 181s 572ms/step - accuracy: 0.9489 - loss: 0.1324 - val_accuracy: 0.8882 - val_loss: 0.2874
Epoch 6/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.9649 - loss: 0.0985
Epoch 6: val_accuracy did not improve from 0.88823
316/316 ━━━━━━━━━━━━━━━━━━━━ 180s 570ms/step - accuracy: 0.9649 - loss: 0.0985 - val_accuracy: 0.8874 - val_loss: 0.3009
Epoch 7/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step - accuracy: 0.9754 - loss: 0.0765
Epoch 7: val_accuracy did not improve from 0.88823
316/316 ━━━━━━━━━━━━━━━━━━━━ 182s 576ms/step - accuracy: 0.9753 - loss: 0.0765 - val_accuracy: 0.8847 - val_loss: 0.3223
Epoch 8/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.9842 - loss: 0.0548
Epoch 8: val_accuracy did not improve from 0.88823
316/316 ━━━━━━━━━━━━━━━━━━━━ 180s 568ms/step - accuracy: 0.9841 - loss: 0.0549 - val_accuracy: 0.8843 - val_loss: 0.3473
Epoch 9/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.9902 - loss: 0.0403
Epoch 9: va

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import uuid
from sklearn.cluster import DBSCAN
import tensorflow as tf
from PIL import Image, ImageChops, ImageEnhance
from io import BytesIO

class DeepUltimateForgeryDetector:
    def __init__(self, model_path='/content/best_ela_model.h5', distance_threshold=100):
        self.distance_threshold = distance_threshold

        # --- 1. YAPAY ZEKA MODELİ (SPLICING İÇİN) ---
        print("Sistem başlatılıyor, yapay zeka modeli yükleniyor...")
        if os.path.exists(model_path):
            self.splicing_model = tf.keras.models.load_model(model_path)
            print("✅ Xception Splicing modeli başarıyla yüklendi!")
        else:
            raise FileNotFoundError(f"HATA: {model_path} bulunamadı!")

        # --- 2. GÖRÜNTÜ İŞLEME ALGORİTMALARI (COPY-MOVE İÇİN) ---
        self.sift = cv2.SIFT_create(nfeatures=10000, contrastThreshold=0.02, edgeThreshold=20)
        self.sift_matcher = cv2.BFMatcher(cv2.NORM_L2)
        self.sift_ratio = 0.65  
        self.sift_weight = 0.60

        self.akaze = cv2.AKAZE_create(threshold=0.0005)
        self.akaze_matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
        self.akaze_ratio = 0.88
        self.akaze_weight = 0.40

    def _load_image(self, image_path):
        try:
            img_array = np.fromfile(image_path, dtype=np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            return img
        except Exception as e:
            print(f"[HATA] Görsel okunamadı: {e}")
            return None

    # =========================================================
    # FEATURE EXTRACTION + MATCHING (COPY-MOVE)
    # =========================================================
    def _extract_and_match(self, gray, detector, matcher, ratio_threshold):
        h, w = gray.shape
        keypoints, descriptors = detector.detectAndCompute(gray, None)

        if descriptors is None or len(descriptors) < 4:
            return [], [], [], []

        matches = matcher.knnMatch(descriptors, descriptors, k=3)
        good_matches = []
        vectors = []

        for match_list in matches:
            if len(match_list) < 3:
                continue

            m, n, o = match_list[0], match_list[1], match_list[2]

            if m.queryIdx == m.trainIdx:
                first = n
                second = o
            else:
                first = m
                second = n

            if first.distance < ratio_threshold * second.distance:
                pt1 = np.array(keypoints[first.queryIdx].pt)
                pt2 = np.array(keypoints[first.trainIdx].pt)

                if pt1[1] < h * 0.82 and pt2[1] < h * 0.82:
                    if np.linalg.norm(pt1 - pt2) > self.distance_threshold:
                        good_matches.append(first)
                        vectors.append(pt2 - pt1)

        return keypoints, descriptors, good_matches, vectors

    def _cluster_vectors(self, matches, vectors):
        if len(vectors) < 3:
            return []
        vectors_np = np.array(vectors)
        clustering = DBSCAN(eps=45, min_samples=2).fit(vectors_np)
        labels = clustering.labels_
        return [matches[i] for i, label in enumerate(labels) if label != -1]

    def _affine_filter(self, keypoints, matches):
        if len(matches) < 3:
            return []
        src_pts = np.float32([keypoints[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([keypoints[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
        try:
            M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=6.0)
            if mask is None:
                return []
            return [matches[i] for i, inlier in enumerate(mask.ravel()) if inlier]
        except:
            return []

    def _process_algorithm(self, gray, detector, matcher, ratio):
        kp, desc, matches, vectors = self._extract_and_match(gray, detector, matcher, ratio)
        if len(matches) == 0:
            return kp, []
        clustered = self._cluster_vectors(matches, vectors)
        final_matches = self._affine_filter(kp, clustered)
        return kp, final_matches

    # =========================================================
    # ELA (Hassaslaştırılmış) + XCEPTION SPLICING
    # =========================================================
    def _generate_ela_for_ai(self, image_path, quality=90):
        """Xception modeli için RAM üzerinde ELA haritası oluşturur."""
        original = Image.open(image_path).convert('RGB')
        buffer = BytesIO()
        original.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        compressed = Image.open(buffer)
        
        ela_image = ImageChops.difference(original, compressed)
        extrema = ela_image.getextrema()
        max_diff = max([ex[1] for ex in extrema]) if extrema else 1
        if max_diff == 0: max_diff = 1
        scale = 255.0 / max_diff
        ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
        
        return ela_image

    def _predict_splicing_with_ai(self, ela_image):
        """ELA haritasını Xception'a besleyip Splicing skorunu (0-100) alır."""
        ela_resized = ela_image.resize((299, 299))
        img_array = np.array(ela_resized) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        prediction = self.splicing_model.predict(img_array, verbose=0)[0][0]
        return prediction * 100.0 # Yüzdelik skor olarak dön

    def _noise_analysis(self, gray):
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        noise = cv2.absdiff(gray, blurred)
        return cv2.normalize(noise, None, 0, 255, cv2.NORM_MINMAX)

    def _edge_analysis(self, gray):
        return cv2.Canny(gray, 100, 200)

    def _create_heatmap(self, img, keypoints, matches):
        if not matches:
            return img
            
        h, w, _ = img.shape
        heatmap_mask = np.zeros((h, w), dtype=np.float32)

        for m in matches:
            pt1 = keypoints[m.queryIdx].pt
            pt2 = keypoints[m.trainIdx].pt
            radius = int(min(h, w) * 0.08)
            cv2.circle(heatmap_mask, (int(pt1[0]), int(pt1[1])), radius, 1.0, -1)
            cv2.circle(heatmap_mask, (int(pt2[0]), int(pt2[1])), radius, 1.0, -1)

        if np.sum(heatmap_mask) > 0:
            heatmap_mask = cv2.GaussianBlur(heatmap_mask, (51, 51), 0)
            heatmap_mask = (heatmap_mask / np.max(heatmap_mask) * 255).astype(np.uint8)
            color_heatmap = cv2.applyColorMap(heatmap_mask, cv2.COLORMAP_JET)
            return cv2.addWeighted(img, 0.6, color_heatmap, 0.4, 0)
        return img

    # =========================================================
    # DETAYLI ORAN ANALİZİ VE SKORLAMA (YAPAY ZEKA ENTEGRELİ)
    # =========================================================
    def _calculate_hybrid_metrics(self, sift_matches, akaze_matches, ai_splicing_score):
        sift_count = len(sift_matches)
        akaze_count = len(akaze_matches)

        # 1. Copy-Move Oranı (Kural Tabanlı - Sizin Mantığınız)
        sift_score = min(sift_count / 25.0, 1.0) if sift_count >= 5 else 0.0
        akaze_score = min(akaze_count / 30.0, 1.0) if akaze_count >= 4 else 0.0
        copy_move_rate = (sift_score * self.sift_weight + akaze_score * self.akaze_weight) * 100
        
        if sift_count < 3 and akaze_count < 3:
            copy_move_rate = 0.0

        # 2. Splicing Oranı (Doğrudan Yapay Zeka Modeli)
        splicing_rate = ai_splicing_score

        # 3. Adli Karar Kapısı
        if sift_count == 0 and akaze_count == 0:
            final_confidence = splicing_rate
        else:
            base_confidence = (copy_move_rate * 0.45) + (splicing_rate * 0.55)
            final_confidence = max(base_confidence, copy_move_rate, splicing_rate)
            
            if sift_count >= 5 and akaze_count >= 4:
                final_confidence += 10.0  # Sinerji Bonusu

        final_confidence = min(max(final_confidence, 0.0), 100.0)
        return copy_move_rate, splicing_rate, final_confidence

    # =========================================================
    # ANA ÇALIŞTIRICI
    # =========================================================
    def detect(self, image_path):
        print("\n" + "=" * 85)
        print("[BAŞLADI] DEEP HYBRID IMAGE FORENSIC ENGINE (AI + RULE BASED)")
        print("=" * 85)

        start = time.time()
        img = self._load_image(image_path)
        if img is None:
            return False

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 1. Kural Tabanlı Süreçler (Copy-Move)
        sift_kp, sift_matches = self._process_algorithm(gray, self.sift, self.sift_matcher, self.sift_ratio)
        akaze_kp, akaze_matches = self._process_algorithm(gray, self.akaze, self.akaze_matcher, self.akaze_ratio)

        # 2. Yapay Zeka Süreçleri (Splicing)
        ela_pil = self._generate_ela_for_ai(image_path)
        ai_splicing_score = self._predict_splicing_with_ai(ela_pil)
        
        # Ek görseller için
        noise = self._noise_analysis(gray)
        edges = self._edge_analysis(gray)

        # 3. Hibrit Metriklerin Hesaplanması
        cm_rate, sp_rate, confidence = self._calculate_hybrid_metrics(sift_matches, akaze_matches, ai_splicing_score)
        elapsed = time.time() - start

        # Konsol Çıktıları
        print(f"[SIFT] Final Match Count: {len(sift_matches)}")
        print(f"[AKAZE] Final Match Count: {len(akaze_matches)}")
        print(f"[METRİK] Copy-Move Sahtecilik Oranı (Algoritmik): %{cm_rate:.2f}")
        print(f"[METRİK] Splicing Ekleme Oranı (Yapay Zeka): %{sp_rate:.2f}")
        print(f"[SONUÇ] Genel Güven/Şüphe Skoru: %{confidence:.2f}")
        print(f"[BİLGİ] Analiz Süresi: {elapsed:.2f} saniye")

        if confidence >= 40:
            print("\n" + "!" * 85)
            print("[ALARM] MANİPÜLASYON / SAHTECİLİK TESPİT EDİLDİ")
            print("!" * 85)
        else:
            print("\n" + "-" * 85)
            print("[SONUÇ] Görsel temiz görünüyor.")
            print("-" * 85)

        # Matplotlib Görselleştirme Paneli (3x2 Subplots)
        fig, axes = plt.subplots(3, 2, figsize=(18, 16))

        # SIFT
        sift_draw = cv2.drawMatches(img, sift_kp, img, sift_kp, sift_matches, None, flags=2)
        axes[0, 0].imshow(cv2.cvtColor(sift_draw, cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title(f"SIFT Matches ({len(sift_matches)})", color='red', fontsize=12)
        axes[0, 0].axis("off")

        # AKAZE
        akaze_draw = cv2.drawMatches(img, akaze_kp, img, akaze_kp, akaze_matches, None, flags=2)
        axes[0, 1].imshow(cv2.cvtColor(akaze_draw, cv2.COLOR_BGR2RGB))
        axes[0, 1].set_title(f"AKAZE Matches ({len(akaze_matches)})", color='red', fontsize=12)
        axes[0, 1].axis("off")

        # SIFT Isı Haritası
        sift_heat = self._create_heatmap(img, sift_kp, sift_matches)
        axes[1, 0].imshow(cv2.cvtColor(sift_heat, cv2.COLOR_BGR2RGB))
        axes[1, 0].set_title("SIFT Regional Localization", color='darkred', fontsize=12)
        axes[1, 0].axis("off")

        # AKAZE Isı Haritası
        akaze_heat = self._create_heatmap(img, akaze_kp, akaze_matches)
        axes[1, 1].imshow(cv2.cvtColor(akaze_heat, cv2.COLOR_BGR2RGB))
        axes[1, 1].set_title("AKAZE Regional Localization", color='darkred', fontsize=12)
        axes[1, 1].axis("off")

        # ELA Haritası (Yapay Zekaya Giden)
        axes[2, 0].imshow(np.array(ela_pil))
        axes[2, 0].set_title(f"AI ELA Map (Splicing Score: %{sp_rate:.2f})", color='purple', fontsize=12)
        axes[2, 0].axis("off")

        # Gürültü ve Kenar Tutarsızlığı
        combined = cv2.addWeighted(noise, 0.7, edges, 0.3, 0)
        axes[2, 1].imshow(combined, cmap='gray')
        axes[2, 1].set_title("Noise + Edge Inconsistency Map", color='purple', fontsize=12)
        axes[2, 1].axis("off")

        report_title = (
            f"Deep Hybrid Forensic Report (AI + Algorithms)\n"
            f"Genel Şüphe Skoru: %{confidence:.2f}  |  "
            f"Copy-Move İhtimali: %{cm_rate:.2f}  |  "
            f"AI Splicing İhtimali: %{sp_rate:.2f}"
        )
        plt.suptitle(report_title, fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.show()

        return confidence >= 40

# --- KULLANIM (COLAB ORTAMI İÇİN) ---
if __name__ == "__main__":
    import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# 1. Modeli Başlat (Yolun doğru olduğundan emin olun)
print("Sistem başlatılıyor, yapay zeka modeli yükleniyor...")
detector = DeepUltimateForgeryDetector(
    model_path='/kaggle/working/best_casia_ela_model.h5', 
    distance_threshold=100
)

# 2. Arayüz Elemanlarını Oluştur
print("\nTest edilecek görseli seçin:")
upload_widget = widgets.FileUpload(
    accept='image/*',  # Sadece resim formatları
    multiple=False     # Tek seferde bir dosya
)

analyze_button = widgets.Button(
    description="Resmi Analiz Et",
    button_style='success', # Butonu yeşil yapar
    tooltip='Yüklenen resmi analiz motoruna gönderir'
)

output_area = widgets.Output()

# Buton ve Yükleme aracını ekranda göster
display(upload_widget, analyze_button, output_area)

# 3. Tıklama Olayı (Event Listener) İşlevi
def on_analyze_clicked(b):
    with output_area:
        clear_output() # Önceki analiz çıktılarını temizle
        
        if upload_widget.value:
            # İpywidgets sürüm uyumluluğu (tuple veya dict dönebilir)
            try:
                # ipywidgets 8.x (Kaggle güncel sürümü)
                if isinstance(upload_widget.value, tuple):
                    file_info = upload_widget.value[0]
                    uploaded_filename = file_info['name']
                    file_content = file_info['content']
                # ipywidgets 7.x (Eski sürüm fallback)
                else:
                    uploaded_filename = list(upload_widget.value.keys())[0]
                    file_content = upload_widget.value[uploaded_filename]['content']
                
                # Dosyayı çalışma dizinine kaydet
                with open(uploaded_filename, 'wb') as f:
                    f.write(file_content)
                
                print(f"\n[BAŞARILI] '{uploaded_filename}' sisteme alındı.")
                print("="*60)
                
                # Tespit motorunu çalıştır
                detector.detect(uploaded_filename)
                
            except Exception as e:
                print(f"[HATA] Beklenmeyen bir sorun oluştu: {e}")
        else:
            print("\n[UYARI] Lütfen önce 'Upload' butonundan bir resim seçin ve yüklenmesini bekleyin!")

# Butonu fonksiyona bağla
analyze_button.on_click(on_analyze_clicked)

Sistem başlatılıyor, yapay zeka modeli yükleniyor...
Sistem başlatılıyor, yapay zeka modeli yükleniyor...


✅ Xception Splicing modeli başarıyla yüklendi!

Test edilecek görseli seçin:


FileUpload(value=(), accept='image/*', description='Upload')

Button(button_style='success', description='Resmi Analiz Et', style=ButtonStyle(), tooltip='Yüklenen resmi ana…

Output()

In [9]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/best_casia_ela_model.h5')

/kaggle/working/best_casia_ela_model.h5